# **Задание №6. Разработка спецификации требований к информационной системе (SRS)**

СПЕЦИФИКАЦИЯ ТРЕБОВАНИЙ К ПРОГРАММНОМУ ОБЕСПЕЧЕНИЮ
Веб-сервис генерации квизов на основе пользовательских документов с использованием локальной LLM и экспорта в PowerPoint

## 1. ВВЕДЕНИЕ

### 1.1. Цель документа

Цель данного документа - формализовать требования к программному обеспечению веб-сервиса генерации квизов на основе загруженных пользователем документов с использованием локальной языковой модели (LLM) и экспорта результатов в формат PowerPoint и JSON. Спецификация предназначена для разработчиков, научного руководителя и потенциальных пользователей, чтобы обеспечить единое понимание функционала системы, ограничений и критериев приемки.

### 1.2. Область применения

Разрабатываемая система предназначена для преподавателей, студентов, ведущих мероприятий и создателей образовательного контента, которым необходимо быстро получать готовые квизы по своим материалам. Пользователь загружает документы (PDF, DOCX, TXT, HTML), система анализирует содержимое с помощью RAG-подхода и локальной LLM, генерирует вопросы, позволяет их отредактировать и экспортировать квиз в виде презентации PowerPoint или JSON-структуры. Система не предназначена для онлайн-прохождения квизов, а служит исключительно генератором материалов.

### 1.3. Определения, акронимы и сокращения

* LLM - Large Language Model, большая языковая модель (в данном проекте - локально развёрнутая модель семейства Llama).
* RAG - Retrieval-Augmented Generation, подход генерации текста с предварительным поиском релевантных фрагментов в векторной базе.
* Квиз - набор вопросов с вариантами ответов и/или открытыми вопросами, формируемый на основе документов.
* Чанк - фрагмент текста, полученный после нарезки исходного документа.
* PPTX - формат презентаций Microsoft PowerPoint.
* Пользователь (автор квиза) - зарегистрированный пользователь, создающий квизы.
* Администратор - пользователь с расширенными правами управления пользователями и квизами.

### 1.4. Пользовательские роли

**Роль 1: Автор квиза (обычный пользователь)**
Описание: зарегистрированный пользователь, создающий и редактирующий квизы на основе собственных документов.
Количество: основная масса пользователей системы.
Технический уровень: базовый или средний уровень владения ПК и веб-приложениями.
Основные задачи: вход в систему, загрузка документов, настройка параметров генерации, запуск генерации, редактирование вопросов, экспорт квизов в PPTX и JSON.

Пример сценариев использования:

* Войти через Google, создать новый квиз, загрузить PDF-файл, задать количество вопросов, сгенерировать квиз, отредактировать формулировки и скачать презентацию.
* Открыть ранее сгенерированный квиз (до истечения срока хранения), внести коррекции и повторно экспортировать его.

**Роль 2: Администратор**
Описание: пользователь, отвечающий за управление системой и модерацию.
Количество: 1–2 человека.
Технический уровень: продвинутый пользователь, владеющий базовыми навыками администрирования систем.
Основные задачи: просмотр списка пользователей, удаление или блокировка проблемных аккаунтов, просмотр и удаление любых квизов, контроль объема хранимых данных.

Пример сценариев использования:

* Просмотреть список всех пользователей и их активных квизов, удалить квизы, нарушающие правила.
* Проверить статистику использования системы и очистить устаревшие данные.

## 2. ФУНКЦИОНАЛЬНЫЕ ТРЕБОВАНИЯ

### 2.1. Требования к аутентификации и учетной записи

**ФТ-001: Авторизация через Google OAuth2**
Описание: система должна обеспечивать вход пользователя с использованием учетной записи Google.
Входные данные: запрос аутентификации от пользователя, ответ от сервиса Google (авторизационный код/токен).
Выходные данные: созданная или найденная запись пользователя в системе, активная сессия.
Бизнес-правила: при первом входе создается новая учетная запись; при повторном входе выполняется привязка по external_id.
Приоритет: Критический.
Критерий проверки: при нажатии кнопки "Войти через Google" пользователь успешно перенаправляется, проходит авторизацию и попадает в личный кабинет без ввода пароля в локальной системе.

**ФТ-002: Отображение личного кабинета пользователя**
Описание: после авторизации система должна отображать личный кабинет с перечнем квизов пользователя и возможностью создать новый.
Входные данные: идентификатор текущего пользователя.
Выходные данные: список квизов (название, дата создания, статус, срок истечения), кнопка создания нового квиза.
Бизнес-правила: в списке отображаются только квизы данного пользователя, не истекшие по сроку хранения.
Приоритет: Высокий.
Критерий проверки: после авторизации отображается список квизов только текущего пользователя и интерфейс создания нового квиза.

### 2.2. Требования к управлению квизами

**ФТ-003: Создание нового квиза**
Описание: система должна позволять пользователю создавать новый квиз, задавая базовые параметры генерации.
Входные данные: название квиза, язык (по умолчанию ru), количество вопросов, стиль вопросов, master prompt (опционально), режим экспорта ответов.
Выходные данные: черновик квиза со статусом draft, привязанный к пользователю.
Бизнес-правила: у пользователя может быть только один активный квиз с незавершенной генерацией; при превышении лимита система предлагает удалить или завершить текущий.
Приоритет: Критический.
Критерий проверки: при заполнении формы и сохранении создается запись квиза в БД, отображаемая в личном кабинете.

**ФТ-004: Просмотр списка квизов пользователя**
Описание: пользователь должен иметь возможность видеть список всех своих квизов.
Входные данные: идентификатор пользователя.
Выходные данные: упорядоченный список квизов с названиями, датами создания, статусами и признаком истечения.
Бизнес-правила: квизы с истекшим сроком могут отображаться как неактивные, но недоступны для генерации новых экспортов.
Приоритет: Высокий.
Критерий проверки: при открытии личного кабинета отображается корректный список квизов пользователя.

**ФТ-005: Удаление квиза пользователем**
Описание: система должна позволять пользователю вручную удалять свои квизы.
Входные данные: идентификатор квиза, принадлежащего пользователю.
Выходные данные: квиз помечается как удаленный, документы и временные файлы подлежат очистке.
Бизнес-правила: удаление квиза необратимо; при удалении удаляются ссылки на связанные документы и чанки.
Приоритет: Средний.
Критерий проверки: после подтверждения удаления квиз пропадает из списка, попытка доступа по прямому ID возвращает ошибку.

### 2.3. Требования к работе с документами и RAG

**ФТ-006: Загрузка документов для квиза**
Описание: система должна позволять загружать один или несколько документов к выбранному квизу.
Входные данные: файлы форматов PDF, DOCX, TXT, HTML размером до 5 МБ каждый.
Выходные данные: записи о документах с путями к файлам в локальном хранилище.
Бизнес-правила: суммарный объем файлов может быть ограничен; формат файла проверяется, некорректные файлы отклоняются с сообщением об ошибке.
Приоритет: Критический.
Критерий проверки: после загрузки документы отображаются в списке материалов квиза, а файлы физически присутствуют в файловой системе.

**ФТ-007: Парсинг документов и создание чанков**
Описание: система должна извлекать текст из загруженных документов, разбивать его на чанки и сохранять в векторной БД.
Входные данные: идентификаторы загруженных документов.
Выходные данные: набор чанков с текстом и embedding-векторами, привязанных к документам.
Бизнес-правила: длина чанка и перекрытие задаются конфигурацией; документы с пустым или некорректным текстом помечаются как проблемные.
Приоритет: Критический.
Критерий проверки: после запуска обработки в базе данных появляются записи чанков, доступные для поиска и генерации вопросов.

### 2.4. Требования к генерации и редактированию квизов

**ФТ-008: Генерация вопросов с использованием LLM**
Описание: система должна уметь генерировать вопросы на основе релевантных чанков с использованием локальной LLM.
Входные данные: идентификатор квиза, параметры генерации (количество вопросов, стиль, master prompt).
Выходные данные: набор вопросов с типами, вариантами ответов и привязкой к квизу.
Бизнес-правила: количество вопросов по умолчанию - 10; вопросы формируются на русском языке; поддерживаются типы single, multiple, open, matching.
Приоритет: Критический.
Критерий проверки: при запуске генерации для квиза с загруженными документами создаются вопросы, которые отображаются в редакторе.

**ФТ-009: Редактирование структуры квиза**
Описание: пользователь должен иметь возможность редактировать сгенерированные вопросы.
Входные данные: выбранный вопрос, измененный текст, список вариантов ответов, флаги правильных ответов, новый порядок и вес.
Выходные данные: обновленные записи вопросов и вариантов ответов.
Бизнес-правила: каждый вопрос должен иметь хотя бы один правильный вариант (для single/multiple); порядок вопросов сохраняется в поле order_index.
Приоритет: Высокий.
Критерий проверки: изменения, внесенные пользователем в интерфейсе, сохраняются и корректно отображаются при повторном открытии редактора.

**ФТ-010: Автосохранение квиза**
Описание: система должна периодически сохранять изменения в квизе без явного нажатия пользователем кнопки "Сохранить".
Входные данные: текущая версия структуры квиза при редактировании.
Выходные данные: обновленная запись квиза и связанных вопросов в БД.
Бизнес-правила: автосохранение выполняется каждые 5 минут или при существенных изменениях; в случае сбоя пользователь не теряет последние правки.
Приоритет: Средний.
Критерий проверки: при отключении браузера и повторном входе изменения за последние 5 минут сохранены.

### 2.5. Требования к экспорту

**ФТ-011: Экспорт квиза в PPTX**
Описание: система должна формировать PowerPoint-презентацию на основе текущей версии квиза.
Входные данные: идентификатор квиза, выбранный режим отображения ответов (в конце или после каждого вопроса).
Выходные данные: PPTX-файл с слайдами, содержащими вопросы и ответы.
Бизнес-правила: один вопрос располагается на одном слайде; ответы либо группируются в конце, либо выводятся на отдельном слайде после вопроса; дизайн - минималистичный.
Приоритет: Критический.
Критерий проверки: после запуска экспорта пользователь получает PPTX-файл, открываемый в PowerPoint, структура соответствует выбранному режиму.

**ФТ-012: Экспорт квиза в JSON**
Описание: система должна предоставлять экспорт структуры квиза в JSON-формате.
Входные данные: идентификатор квиза.
Выходные данные: JSON-документ с полями title, language, created_at, списком вопросов и вариантов ответов, метаданными.
Бизнес-правила: формат JSON должен быть стабильным и пригодным для повторного импорта в будущем.
Приоритет: Высокий.
Критерий проверки: при вызове функции экспорта пользователь получает корректный JSON-файл, который проходит проверку парсером без ошибок.

### 2.6. Требования к административному функционалу

**ФТ-013: Просмотр и управление пользователями (администратор)**
Описание: администратор должен иметь возможность просматривать список всех пользователей и управлять их статусом.
Входные данные: запрос от пользователя с правами администратора.
Выходные данные: список пользователей с email, датой регистрации, количеством квизов, флагом is_admin и возможностью блокировки/удаления.
Бизнес-правила: удаление пользователя может быть запрещено, если это нарушает целостность данных; возможна деактивация вместо физического удаления.
Приоритет: Средний.
Критерий проверки: при входе под администратором доступна отдельная страница с полным списком пользователей и действиями над ними.

**ФТ-014: Просмотр и удаление квизов пользователей (администратор)**
Описание: администратор должен иметь возможность просматривать и удалять любые квизы.
Входные данные: идентификатор квиза, выбранный администратором.
Выходные данные: измененный статус квиза (удален), запуск процедуры очистки файлов.
Бизнес-правила: действия администратора логируются; удаление квиза пользователя не должно приводить к сбою системы.
Приоритет: Средний.
Критерий проверки: администратор может открыть список квизов по пользователям и удалить любой из них, после чего квиз недоступен ни владельцу, ни администратору.



## 3. НЕФУНКЦИОНАЛЬНЫЕ ТРЕБОВАНИЯ

### 3.1. Требования к производительности

**НФТ-П-001: Время отклика пользовательского интерфейса**
Система должна обеспечивать время отклика основных операций (загрузить страницу личного кабинета, открыть редактор квиза, просмотреть список вопросов) не более 3 секунд при размере документов до 5 МБ и до 10 одновременных активных пользователей.
Обоснование: значение выбрано для комфортной работы пользователя в условиях учебного проекта с небольшим числом пользователей.

**НФТ-П-002: Время генерации квиза**
Время генерации квиза из документов суммарным размером до 5 МБ не должно превышать 60 секунд в 95% случаев при наличии доступного GPU.
Обоснование: генерация с участием LLM может быть ресурсозатратной, поэтому выбран разумный компромисс между скоростью и качеством.

### 3.2. Требования к надежности

**НФТ-Н-001: Восстановление после сбоя**
После перезапуска сервера система должна корректно восстановить работоспособность без потери сохраненных квизов и документов, за исключением данных, помеченных для автоудаления по истечении срока хранения.

**НФТ-Н-002: Обработка ошибок парсинга и генерации**
При ошибках парсинга документов или генерации вопросов система не должна аварийно завершать работу, а должна возвращать пользователю понятное сообщение и сохранять информацию об ошибке в журнале.

### 3.3. Требования к безопасности

**НФТ-Б-001: Разграничение доступа к данным**
Пользователь не должен иметь доступа к квизам и документам других пользователей; все операции чтения и изменения данных должны проверять принадлежность ресурса текущему пользователю.

**НФТ-Б-002: Локальная обработка документов и модели**
Документы и текст, извлеченный из них, не должны отправляться во внешние облачные сервисы для обработки; LLM должна быть развернута в контролируемой локальной или арендованной инфраструктуре.
Обоснование: это снижает риск утечки конфиденциальных данных из загруженных документов.

### 3.4. Требования к удобству использования

**НФТ-Ю-001: Двуязычный интерфейс**
Пользовательский интерфейс должен поддерживать русский и английский языки с возможностью переключения языка не более чем в два действия.

**НФТ-Ю-002: Число шагов для создания квиза**
Создание нового квиза, загрузка документов и запуск генерации должны укладываться не более чем в 5 последовательных экранов.
Обоснование: это упрощает работу для пользователей без технического бэкграунда.

### 3.5. Требования к совместимости

**НФТ-С-001: Поддерживаемые браузеры**
Веб-интерфейс должен корректно работать в актуальных версиях браузеров Google Chrome, Mozilla Firefox и Microsoft Edge (последние 2 стабильные версии).

**НФТ-С-002: Серверная среда**
Серверная часть должна быть совместима с ОС Linux x86_64, Python версии не ниже 3.11 и PostgreSQL версии не ниже 15.

### 3.6. Требования к масштабируемости

**НФТ-М-001: Масштабируемость API**
Серверное API должно быть спроектировано как статeless (без сохранения состояния сессий на уровне приложения), что позволяет использовать горизонтальное масштабирование при необходимости.

**НФТ-М-002: Вынесение LLM в отдельный сервис**
Архитектура должна предусматривать возможность вынесения LLM на отдельный сервис или узел с конфигурируемым адресом, не затрагивая бизнес-логику приложения.
Обоснование: это позволит при росте нагрузки заменить локальный инстанс модели на более мощный без переписывания всего приложения.

## 4. ТРЕБОВАНИЯ К ИНТЕРФЕЙСАМ

### 4.1. Пользовательский интерфейс

Основные экраны:

* Экран авторизации: кнопка "Войти через Google", переключение языка.
* Личный кабинет: список квизов, кнопка "Создать квиз", индикатор статуса и срока хранения.
* Мастер создания квиза: форма параметров (название, язык, количество вопросов, стиль, master prompt, режим экспорта), интерфейс загрузки документов.
* Редактор квиза: список всех вопросов с возможностью редактирования текста, вариантов ответов, порядка, веса и флагов правильности.
* Экран экспорта: выбор режима PPTX (ответы в конце или после каждого вопроса), кнопка "Скачать PPTX" и "Скачать JSON".
* Админ-панель: список пользователей, список квизов, действия по удалению/блокировке.

### 4.2. Программный интерфейс (API)

Система должна предоставлять REST API, включающее следующие группы эндпоинтов (без детальной спецификации запросов):

* /auth/google - обработка OAuth2-авторизации.
* /users/me - получение информации о текущем пользователе.
* /quizzes - создание и получение списка квизов.
* /quizzes/{id} - получение, обновление и удаление квиза.
* /quizzes/{id}/documents - загрузка и список связанных документов.
* /quizzes/{id}/generate - запуск генерации вопросов.
* /quizzes/{id}/questions - чтение и запись структуры вопросов.
* /quizzes/{id}/export/pptx - экспорт в PPTX.
* /quizzes/{id}/export/json - экспорт в JSON.
* /admin/users, /admin/quizzes - административные операции.

### 4.3. Интерфейсы с внешними системами

* Google OAuth2 - для аутентификации пользователей.
* Сервис LLM (локальный или выделенный сервер) - для генерации вопросов по текстовым чанкам.


## 5. ОГРАНИЧЕНИЯ И ДОПУЩЕНИЯ

### 5.1. Ограничения

**ОГР-001:** Система должна использовать PostgreSQL в качестве основной реляционной СУБД, с расширением pgvector для хранения векторных представлений.

**ОГР-002:** Документы пользователей хранятся только в локальной файловой системе сервера; использование облачных хранилищ в рамках дипломного проекта не предусматривается.

**ОГР-003:** Поддерживаются только форматы PDF, DOCX, TXT и HTML, размер каждого файла ограничен 5 МБ.

**ОГР-004:** У каждого пользователя может быть не более одного активного квиза в стадии генерации, срок хранения данных квиза и документов - до 3 дней.

### 5.2. Допущения

**ДОП-001:** Пользователь имеет стабильное интернет-соединение и использует современные версии поддерживаемых браузеров.

**ДОП-002:** Число одновременных активных пользователей не превышает 10, что соответствует масштабу учебного проекта.

**ДОП-003:** Качество загружаемых документов (отсканированных или цифровых) позволяет корректно извлекать из них текст.

## 6. КРИТЕРИИ ПРИЕМКИ

### 6.1. Функциональная приемка

**КП-Ф-001:** Пользователь может войти через Google, создать новый квиз, загрузить документы, сгенерировать не менее 10 вопросов, отредактировать их и успешно экспортировать квиз в PPTX и JSON.

**КП-Ф-002:** Администратор может просмотреть список пользователей и квизов, удалить квиз любого пользователя, и этот квиз становится недоступным для владельца.

### 6.2. Нефункциональная приемка

**КП-НФ-001:** При тестировании на наборе документов суммарным объемом до 5 МБ время генерации квиза не превышает 60 секунд в 9 из 10 попыток.

**КП-НФ-002:** При одновременной работе 5 тестовых пользователей не наблюдается ошибок HTTP 5xx, а время отклика основных экранов не превышает 3 секунд.

### 6.3. Документация

**КП-Д-001:** Подготовлена пояснительная записка, содержащая описание архитектуры, ER-диаграммы, UML-диаграммы и описание основных сценариев работы системы.

**КП-Д-002:** Подготовлено краткое руководство пользователя, описывающее процесс входа, создания квиза, загрузки документов, генерации и экспорта.

### 6.4. Тестирование

**КП-Т-001:** Реализован набор модульных тестов для ключевых модулей (обработка документов, генерация вопросов, экспорт), все тесты выполняются успешно.

**КП-Т-002:** Проведен ручной прогон основных сценариев использования (happy-path), результаты зафиксированы в виде чек-листа, все критические замечания устранены.